# Stablecoin Depegging Recovery Speed Prediction
**MSc Business Analytics Dissertation — Complete Pipeline**
Yi-Wen Su | MSc Business Analytics | Alliance Manchester Business School

---
**Pipeline overview**
- Stage 0: Data loading & event detection
- Stage 1: Feature engineering (33 candidate features, measured at t-1)
- Stage 2: Multicollinearity removal (r > 0.7, theory-based) — outside CV
- Stage 3: Crisis group assignment (for group-aware CV)
- Stage 4: Nested Cross-Validation (outer: repeated stratified 5-fold × 5 seeds; inner: 3-fold)
- Stage 5: Robustness checks (group-aware CV / 7-day cutoff / excl. zero-slow)
- Stage 6: EDA & descriptive statistics
- Stage 7: Save all outputs

**Data sources:** CoinGecko CSV exports (USDT, USDC, BUSD, DAI, FRAX, TUSD, USDD, USTC, LUSD; BTC; ETH)

**Before running:** place `stablecoin_events_dataset.csv` in the same folder as this notebook.


In [1]:
# Run this once if you get ModuleNotFoundError (e.g. imbalanced-learn / scipy)
# %pip install imbalanced-learn scipy scikit-learn pandas numpy

In [2]:
"""
Stablecoin Depegging Recovery Speed Prediction
MSc Business Analytics Dissertation — Complete Pipeline
========================================================
Yi-Wen Su | MSc Business Analytics | Alliance Manchester Business School

PIPELINE OVERVIEW
=================
Stage 0: Data loading & event detection
Stage 1: Feature engineering (33 candidate features, measured at t-1)
Stage 2: Multicollinearity removal (r > 0.7, theory-based)  [outside CV]
Stage 3: Nested Cross-Validation
          Outer: Repeated Stratified 5-fold × 5 seeds = 25 folds
          Inner: 3-fold CV for (a) top-N feature selection via RF
                              (b) hyperparameter tuning
Stage 4: SMOTE sensitivity analysis
Stage 5: Robustness checks (group-aware CV / 7-day cutoff / excl. zero-slow)
Stage 6: EDA & descriptive statistics
Stage 7: Save all outputs

DATA SOURCES
============
Stablecoin price data : CoinGecko (CSV exports)
  USDT, USDC, BUSD, DAI, FRAX, TUSD, USDD, USTC, LUSD
BTC + ETH price data  : CoinGecko (CSV exports)

DATA CUTOFFS
============
BUSD : truncated 2023-02-13 (NYDFS order to Paxos to cease minting)
USTC : truncated 2022-05-09 (day before Terra collapse)
       Collapse event (2022-05-10 onwards) excluded as permanent depeg

LABEL DEFINITION
================
Depegging threshold : ±2% from $1 peg (daily closing price)
Recovery cutoff     : 3 days (P75 of empirical duration distribution)
  slow (label=1)    : duration > 3 days
  fast (label=0)    : duration ≤ 3 days
"""

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (StratifiedKFold, GridSearchCV,
                                     RepeatedStratifiedKFold, GroupKFold)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, accuracy_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings('ignore')

## STAGE 0: LOAD DATA

In [4]:
"""
Input file: stablecoin_events_dataset.csv
  This file was produced by the event detection script (not shown here).
  Each row = one depegging event.
  Key columns:
    coin            : stablecoin name
    type            : mechanism type (Bullmann et al. 2019)
    event_start     : first day price crossed ±2% boundary
    event_end       : last day price remained outside boundary
    duration_days   : event_end - event_start + 1
    initial_deviation_pct : max |price - 1| × 100 on event_start day
    direction       : 'below' (price < $0.98) or 'above' (price > $1.02)
    label           : 'fast' (≤3d) or 'slow' (>3d)
    [feature columns]: all measured at t-1 (day before event_start)
"""

df = pd.read_csv('../data/stablecoin_events_dataset.csv')
df['event_start'] = pd.to_datetime(df['event_start'])
df['y']    = (df['label'] == 'slow').astype(int)   # slow=1, fast=0
df['y_7d'] = (df['duration_days'] > 7).astype(int) # for robustness check

print("=" * 65)
print("STABLECOIN RECOVERY SPEED — Complete Pipeline")
print("=" * 65)
print(f"\nDataset: n={len(df)}  "
      f"slow={df['y'].sum()} ({df['y'].mean()*100:.1f}%)  "
      f"fast={(df['y']==0).sum()} ({(df['y']==0).mean()*100:.1f}%)")
print(f"Coins: {sorted(df['coin'].unique().tolist())}")
print(f"Date range: {df['event_start'].min().date()} → "
      f"{df['event_start'].max().date()}")

STABLECOIN RECOVERY SPEED — Complete Pipeline

Dataset: n=135  slow=28 (20.7%)  fast=107 (79.3%)
Coins: ['BUSD', 'DAI', 'FRAX', 'LUSD', 'TUSD', 'USDC', 'USDD', 'USDT', 'USTC']
Date range: 2015-02-25 → 2026-03-30


## STAGE 1: FEATURE DEFINITION 33 candidate features across 5 groups (A–E) All measured at t-1 (day before event_start) to prevent look-ahead bias

In [5]:
ORIGINAL_33 = [
    # ── Group A: Event characteristics ──────────────────────────
    # Measured at event onset (not t-1); describe the event itself
    'initial_deviation_pct',   # H2: max |price-1|×100 on event start day
    'direction_below',         # H2: 1=below $1, 0=above $1

    # ── Group B: Mechanism type dummies (Bullmann et al. 2019) ──
    # H1: mechanism type → recovery speed
    'type_tokenised',          # Tokenised Fund: USDT, USDC, TUSD
    'type_offchain',           # Off-chain Collateral: BUSD
    'type_onchain',            # On-chain Collateral: DAI, LUSD
    'type_algo',               # Algorithmic: FRAX, USDD, USTC

    # ── Group C: SC market dynamics (t-1) ───────────────────────
    # H3: market liquidity → recovery speed
    'sc_ret_1d',               # SC log return 1d
    'sc_ret_7d',               # SC log return 7d
    'sc_ret_30d',              # SC log return 30d (missing if <30d history)
    'sc_vol_chg_1d',           # SC volume log change 1d
    'sc_vol_chg_7d',           # SC volume log change 7d
    'sc_vol_chg_30d',          # SC volume log change 30d
    'sc_mcap_chg_1d',          # SC market cap log change 1d
    'sc_mcap_chg_7d',          # SC market cap log change 7d
    'sc_vol_ratio',            # Volume / Market cap (liquidity proxy)

    # ── Group D: SC volatility (t-1) ────────────────────────────
    # H5: pre-event stability → recovery speed
    'sc_vol_5d',               # SC annualised rolling vol 5d
    'sc_vol_30d',              # SC annualised rolling vol 30d
    'sc_price_dev_5d',         # RMS deviation from $1 over 5d (Kwon 2023)
    'sc_price_dev_30d',        # RMS deviation from $1 over 30d
    'sc_dw_dev_5d',            # Downward RMS deviation from $1 over 5d
    'sc_dw_dev_30d',           # Downward RMS deviation from $1 over 30d

    # ── Group E: BTC market conditions (t-1) ────────────────────
    # H4: crypto market stress → recovery speed
    'btc_ret_1d',              # BTC log return 1d
    'btc_ret_7d',              # BTC log return 7d
    'btc_ret_30d',             # BTC log return 30d
    'btc_vol_chg_7d',          # BTC volume log change 7d
    'btc_vol_chg_30d',         # BTC volume log change 30d
    'btc_realized_vol',        # BTC annualised rolling vol 30d

    # ── Group E: ETH market conditions (t-1) ────────────────────
    # H4: crypto market stress → recovery speed
    'eth_ret_1d',              # ETH log return 1d
    'eth_ret_7d',              # ETH log return 7d
    'eth_ret_30d',             # ETH log return 30d
    'eth_vol_chg_7d',          # ETH volume log change 7d
    'eth_mcap_chg_7d',         # ETH market cap log change 7d
    'eth_realized_vol',        # ETH annualised rolling vol 30d
]

# Impute missing values (median): arises from coins with <30d history
# at time of early events — not event-period contamination
df[ORIGINAL_33] = df[ORIGINAL_33].fillna(df[ORIGINAL_33].median())
print(f"\nStage 1: {len(ORIGINAL_33)} candidate features defined across 5 groups")


Stage 1: 33 candidate features defined across 5 groups


## STAGE 2: MULTICOLLINEARITY REMOVAL Criterion: Pearson r > 0.7 (Hair et al. 2019) Assessed on full dataset (correlation is a feature-space property, not outcome-dependent → no leakage; Kuhn & Johnson 2013) In all BTC/ETH pairs, ETH retained: majority of sampled stablecoins (DAI, LUSD, USDD, FRAX) are Ethereum-native protocols

In [6]:
print("\n" + "=" * 65)
print("STAGE 2: Multicollinearity removal (r > 0.7)")
print("=" * 65)

corr = df[ORIGINAL_33].corr()
high_pairs = []
for i in range(len(ORIGINAL_33)):
    for j in range(i+1, len(ORIGINAL_33)):
        r = corr.iloc[i,j]
        if abs(r) > 0.7:
            high_pairs.append((ORIGINAL_33[i], ORIGINAL_33[j], round(r,3)))
high_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print(f"\nHigh-correlation pairs found (r > 0.7): {len(high_pairs)}")
for a, b, r in high_pairs:
    print(f"  {a:<30} ↔ {b:<30} r={r:.3f}")

# Removal decisions (theory-based, not result-based)
REMOVED = {
    'eth_mcap_chg_7d':  'r=0.999 with eth_ret_7d: ETH market cap change ≈ '
                        'price return (circulating supply near-constant short-term)',
    'sc_vol_30d':       'r=0.889 with sc_price_dev_30d: retain sc_price_dev_30d '
                        '(peg-specific deviation measure, more theoretically relevant)',
    'btc_realized_vol': 'r=0.882 with eth_realized_vol: both measure market stress; '
                        'ETH retained (Ethereum-native stablecoins in sample)',
    'btc_ret_7d':       'r=0.767 with eth_ret_7d: both measure weekly market return; '
                        'ETH retained (Ethereum-native stablecoins in sample)',
    'btc_ret_30d':      'r=0.765 with eth_ret_30d: both measure monthly market return; '
                        'ETH retained (same reasoning)',
    'btc_ret_1d':       'r=0.728 with eth_ret_1d: both measure daily market return; '
                        'ETH retained (same reasoning)',
    'type_tokenised':   'Reference category for dummy variable set (avoids dummy trap)',
}

print(f"\nRemoved ({len(REMOVED)}):")
for feat, reason in REMOVED.items():
    print(f"  {feat}: {reason}")

# Final candidate pool: 33 - 7 = 26 features
CANDIDATE_POOL = [f for f in ORIGINAL_33 if f not in REMOVED]
print(f"\nCandidate pool after removal: {len(CANDIDATE_POOL)} features")
print(f"  {CANDIDATE_POOL}")

# Verify no remaining pairs > 0.7
corr2 = df[CANDIDATE_POOL].corr()
remaining = [(CANDIDATE_POOL[i], CANDIDATE_POOL[j],
              round(corr2.iloc[i,j],3))
             for i in range(len(CANDIDATE_POOL))
             for j in range(i+1, len(CANDIDATE_POOL))
             if abs(corr2.iloc[i,j]) > 0.7]
if remaining:
    print(f"\n  ⚠ Remaining high-corr pairs: {remaining}")
else:
    print(f"\n  ✓ No remaining pairs exceed r=0.7")

X_full = df[CANDIDATE_POOL].values
y      = df['y'].values


STAGE 2: Multicollinearity removal (r > 0.7)

High-correlation pairs found (r > 0.7): 8
  eth_ret_7d                     ↔ eth_mcap_chg_7d                r=0.999
  sc_vol_30d                     ↔ sc_price_dev_30d               r=0.889
  btc_realized_vol               ↔ eth_realized_vol               r=0.882
  btc_ret_7d                     ↔ eth_mcap_chg_7d                r=0.768
  btc_ret_7d                     ↔ eth_ret_7d                     r=0.767
  btc_ret_30d                    ↔ eth_ret_30d                    r=0.765
  btc_ret_1d                     ↔ eth_ret_1d                     r=0.728
  sc_vol_5d                      ↔ sc_vol_30d                     r=0.716

Removed (7):
  eth_mcap_chg_7d: r=0.999 with eth_ret_7d: ETH market cap change ≈ price return (circulating supply near-constant short-term)
  sc_vol_30d: r=0.889 with sc_price_dev_30d: retain sc_price_dev_30d (peg-specific deviation measure, more theoretically relevant)
  btc_realized_vol: r=0.882 with eth_realized_v

## STAGE 3: CRISIS GROUP ASSIGNMENT Used in Robustness Check 1 (group-aware CV) Events in the same systemic crisis window → same fold

In [7]:
def assign_group(row):
    """Assign crisis group label for group-aware CV."""
    d = row['event_start']
    if pd.Timestamp('2020-03-01') <= d <= pd.Timestamp('2020-03-31'):
        return 'crisis_black_thursday'
    if pd.Timestamp('2022-05-01') <= d <= pd.Timestamp('2022-09-30'):
        return 'crisis_crypto_winter'
    if pd.Timestamp('2022-11-01') <= d <= pd.Timestamp('2022-12-31'):
        return 'crisis_ftx'
    if pd.Timestamp('2023-03-01') <= d <= pd.Timestamp('2023-03-31'):
        return 'crisis_svb'
    return f"coin_{row['coin']}"

groups = LabelEncoder().fit_transform(df.apply(assign_group, axis=1))

In [9]:
df['crisis_group_label'] = df.apply(assign_group, axis=1)
print(df['crisis_group_label'].value_counts())

crisis_group_label
coin_USDT                24
coin_LUSD                20
coin_DAI                 18
coin_TUSD                14
crisis_crypto_winter     13
coin_USDC                10
coin_USDD                 7
coin_USTC                 7
crisis_svb                6
coin_FRAX                 6
crisis_black_thursday     5
crisis_ftx                4
coin_BUSD                 1
Name: count, dtype: int64


## STAGE 4: NESTED CROSS-VALIDATION Structure: Outer loop: Repeated Stratified 5-fold × 5 seeds = 25 evaluations Inner loop: Stratified 3-fold (within each outer train set) Each outer fold independently: (a) RF importance ranking on train set → rank 26 features (b) Top-N selection: try N=1..12, pick N maximising inner-CV F1 (c) Hyperparameter tuning: GridSearch on selected N features (d) Final model: train on full outer train set, evaluate on test set Class imbalance: class_weight='balanced' (primary) SMOTE k=3 (sensitivity analysis, Stage 5)

In [14]:
LR_C_GRID = [0.01, 0.1, 1, 10, 100]
RF_GRID   = {
    'n_estimators':    [100, 200],
    'max_depth':       [1, 2],
    'min_samples_leaf':[10, 15],
    "max_features": ["sqrt"]
}
MAX_N = 12   # max features to evaluate in top-N search

def run_single_fold(X_tr, y_tr, X_te, y_te, mtype, seed, use_smote=False):
    """
    Run one outer fold of nested CV.
    Returns dict of test-set metrics and selected features.
    """
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    sc       = MinMaxScaler().fit(X_tr)
    X_tr_s   = sc.transform(X_tr)

    # ── Step (a): RF importance ranking ─────────────────────────
    rf_rank = RandomForestClassifier(
        n_estimators=50, max_depth=3, min_samples_leaf=5,
        class_weight='balanced', random_state=seed
    ).fit(X_tr_s, y_tr)

    fi = sorted(zip(CANDIDATE_POOL, rf_rank.feature_importances_),
                key=lambda x: x[1], reverse=True)

    # ── Step (b): Top-N selection via inner CV ───────────────────
    best_n, best_inner_f1 = 1, -1
    best_C, best_depth, best_leaf = 10, 1, 5

    for n in range(1, min(MAX_N + 1, len(CANDIDATE_POOL) + 1)):
        fidx = [CANDIDATE_POOL.index(f) for f, _ in fi[:n]]
        X_n  = X_tr_s[:, fidx]

        if mtype == 'LR':
            from sklearn.model_selection import cross_val_score
            for C in LR_C_GRID:
                m = LogisticRegression(
                    C=C, penalty='l2', solver='liblinear',
                    class_weight='balanced', max_iter=300, random_state=seed)
                s = cross_val_score(m, X_n, y_tr, cv=inner_cv,
                                    scoring='f1').mean()
                if s > best_inner_f1:
                    best_inner_f1, best_n, best_C = s, n, C
        else:
            from sklearn.model_selection import cross_val_score
            for depth in [1, 2]:
                for leaf in [10, 15]:
                    m = RandomForestClassifier(
                        n_estimators=50, max_depth=depth,
                        min_samples_leaf=leaf,
                        class_weight='balanced', random_state=seed)
                    s = cross_val_score(m, X_n, y_tr, cv=inner_cv,
                                        scoring='f1').mean()
                    if s > best_inner_f1:
                        best_inner_f1 = s
                        best_n, best_depth, best_leaf = n, depth, leaf

    # ── Step (c): Train final model on selected features ─────────
    fidx     = [CANDIDATE_POOL.index(f) for f, _ in fi[:best_n]]
    selected = [f for f, _ in fi[:best_n]]
    X_tr_f   = X_tr_s[:, fidx]
    X_te_f   = sc.transform(X_te)[:, fidx]

    # Optional SMOTE on train set only
    if use_smote:
        k = min(3, y_tr.sum() - 1)
        if k >= 1:
            sm = SMOTE(random_state=seed, k_neighbors=k)
            X_tr_f, y_tr = sm.fit_resample(X_tr_f, y_tr)

    if mtype == 'LR':
        model = LogisticRegression(
            C=best_C, penalty='l2', solver='liblinear',
            class_weight='balanced', max_iter=300, random_state=seed
        ).fit(X_tr_f, y_tr)
    else:
        model = RandomForestClassifier(
            n_estimators=100, max_depth=best_depth,
            min_samples_leaf=best_leaf,max_features="sqrt",
            class_weight='balanced', random_state=seed
        ).fit(X_tr_f, y_tr)

    # ── Step (d): Evaluate on held-out test set ──────────────────
    yp  = model.predict(X_te_f)
    ypr = model.predict_proba(X_te_f)[:, 1]

    return {
        'f1':           round(f1_score(y_te, yp, zero_division=0), 3),
        'recall':       round(recall_score(y_te, yp, zero_division=0), 3),
        'precision':    round(precision_score(y_te, yp, zero_division=0), 3),
        'auc':          round(roc_auc_score(y_te, ypr), 3),
        'accuracy':     round(accuracy_score(y_te, yp), 3),
        'n_selected':   best_n,
        'selected':     selected,
        'n_test':       len(y_te),
        'n_slow_test':  int(y_te.sum()),
    }


def run_nested_cv(X, y, mtype='LR', use_smote=False,
                  outer_cv=None, groups_arr=None,
                  label=''):
    """
    Run full nested CV across all outer folds.
    Returns (results_list, fold_features_list).
    """
    results, fold_feats = [], []
    iterator = (outer_cv.split(X, y, groups_arr)
                if groups_arr is not None
                else outer_cv.split(X, y))

    for fold_i, (tr_idx, te_idx) in enumerate(iterator):
        r = run_single_fold(
            X[tr_idx], y[tr_idx], X[te_idx], y[te_idx],
            mtype=mtype, seed=fold_i * 7 + 42, use_smote=use_smote
        )
        results.append(r)
        fold_feats.append(r['selected'])

    return results, fold_feats


def summarise(results, label=''):
    """Compute mean ± std across folds."""
    f1s  = [r['f1']  for r in results]
    recs = [r['recall'] for r in results]
    pres = [r['precision'] for r in results]
    aucs = [r['auc']  for r in results]
    ns   = [r['n_selected'] for r in results]
    return {
        'label':        label,
        'n_folds':      len(results),
        'f1_mean':      round(np.mean(f1s), 3),
        'f1_std':       round(np.std(f1s), 3),
        'recall_mean':  round(np.mean(recs), 3),
        'recall_std':   round(np.std(recs), 3),
        'prec_mean':    round(np.mean(pres), 3),
        'auc_mean':     round(np.mean(aucs), 3),
        'auc_std':      round(np.std(aucs), 3),
        'avg_n_feats':  round(np.mean(ns), 1),
    }


# ── Primary: Repeated Stratified 5-fold × 5 seeds ────────────────────────────
print("\n" + "=" * 65)
print("STAGE 4: PRIMARY RESULTS — Nested CV (5-fold × 5 seeds = 25 folds)")
print("=" * 65)

outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

primary_results = {}
primary_feats   = {}

configs = [
    ('LR  class_weight', 'LR',  False),
    ('RF  class_weight', 'RF',  False),
    ('LR  + SMOTE',      'LR',  True),
    ('RF  + SMOTE',      'RF',  True),
]

print(f"\n{'Model':<22} {'F1 mean':>8} {'±std':>6} "
      f"{'Recall':>8} {'AUC':>8} {'N feats':>8}")
print("-" * 60)

all_summaries = []
for name, mtype, smote in configs:
    res, ffeats = run_nested_cv(
        X_full, y, mtype=mtype, use_smote=smote,
        outer_cv=outer_cv, label=name)
    primary_results[name] = res
    primary_feats[name]   = ffeats
    s = summarise(res, label=name)
    all_summaries.append(s)
    print(f"  {name:<20} {s['f1_mean']:>8.3f} {s['f1_std']:>6.3f} "
          f"{s['recall_mean']:>8.3f} {s['auc_mean']:>8.3f} "
          f"{s['avg_n_feats']:>8.1f}")

# ── Feature selection frequency ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("FEATURE SELECTION FREQUENCY (LR class_weight, 25 folds)")
print("Proportion of folds in which each feature was selected")
print("=" * 65)

lr_feats = primary_feats['LR  class_weight']
total_folds = len(lr_feats)
freq = {}
for fold_list in lr_feats:
    for f in fold_list:
        freq[f] = freq.get(f, 0) + 1

freq_df = pd.DataFrame(
    sorted(freq.items(), key=lambda x: x[1], reverse=True),
    columns=['feature', 'count']
)
freq_df['pct'] = (freq_df['count'] / total_folds * 100).round(1)

print(f"\n{'Feature':<32} {'Count':>6} {'%':>6}  Bar")
print("-" * 60)
for _, row in freq_df.iterrows():
    bar = '█' * int(row['pct'] / 5)
    print(f"  {row['feature']:<30} {row['count']:>5}  "
          f"{row['pct']:>5.0f}%  {bar}")


STAGE 4: PRIMARY RESULTS — Nested CV (5-fold × 5 seeds = 25 folds)

Model                   F1 mean   ±std   Recall      AUC  N feats
------------------------------------------------------------
  LR  class_weight        0.415  0.165    0.593    0.697      1.7
  RF  class_weight        0.402  0.165    0.544    0.669      3.0
  LR  + SMOTE             0.420  0.189    0.573    0.699      1.7
  RF  + SMOTE             0.408  0.139    0.589    0.681      3.0

FEATURE SELECTION FREQUENCY (LR class_weight, 25 folds)
Proportion of folds in which each feature was selected

Feature                           Count      %  Bar
------------------------------------------------------------
  initial_deviation_pct             25    100%  ████████████████████
  sc_price_dev_5d                    5     20%  ████
  eth_realized_vol                   4     16%  ███
  sc_vol_5d                          3     12%  ██
  sc_ret_7d                          2      8%  █
  sc_price_dev_30d                   1 

In [19]:
# ── STAGE 4B: BASELINE — Univariate LR (initial_deviation_pct only) ──
# Uses the SAME outer_cv splits as the primary results (Cell above),
# so F1/AUC are directly comparable to the 26-feature pipeline.
print("\n" + "=" * 65)
print("STAGE 4B: BASELINE — Univariate LR (initial_deviation_pct only)")
print("=" * 65)

X_uni = df[['initial_deviation_pct']].values

uni_results = []
for fold_i, (tr_idx, te_idx) in enumerate(outer_cv.split(X_uni, y)):
    X_tr, X_te = X_uni[tr_idx], X_uni[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    sc = MinMaxScaler().fit(X_tr)
    X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)

    model = LogisticRegression(
        class_weight='balanced', max_iter=300, random_state=fold_i * 7 + 42
    ).fit(X_tr_s, y_tr)

    yp  = model.predict(X_te_s)
    ypr = model.predict_proba(X_te_s)[:, 1]

    uni_results.append({
        'f1':         round(f1_score(y_te, yp, zero_division=0), 3),
        'recall':     round(recall_score(y_te, yp, zero_division=0), 3),
        'precision':  round(precision_score(y_te, yp, zero_division=0), 3),
        'auc':        round(roc_auc_score(y_te, ypr), 3),
        'n_selected': 1,
    })

uni_summary = summarise(uni_results, label='Univariate LR (initial_deviation_pct)')
all_summaries.append(uni_summary)   # 直接併入 Table 7 的資料來源

print(f"\n{'Model':<35} {'F1 mean':>8} {'±std':>6} {'Recall':>8} {'AUC':>8}")
print("-" * 70)
print(f"  {uni_summary['label']:<33} {uni_summary['f1_mean']:>8.3f} "
      f"{uni_summary['f1_std']:>6.3f} {uni_summary['recall_mean']:>8.3f} "
      f"{uni_summary['auc_mean']:>8.3f}")
print(f"  {'LR  class_weight (26-feature, full)':<33} "
      f"{all_summaries[0]['f1_mean']:>8.3f} {all_summaries[0]['f1_std']:>6.3f} "
      f"{all_summaries[0]['recall_mean']:>8.3f} {all_summaries[0]['auc_mean']:>8.3f}")
print(f"\n  F1 difference (univariate − full): "
      f"{uni_summary['f1_mean'] - all_summaries[0]['f1_mean']:+.3f}")
print(uni_summary)


STAGE 4B: BASELINE — Univariate LR (initial_deviation_pct only)

Model                                F1 mean   ±std   Recall      AUC
----------------------------------------------------------------------
  Univariate LR (initial_deviation_pct)    0.314  0.184    0.299    0.765
  LR  class_weight (26-feature, full)    0.415  0.165    0.593    0.697

  F1 difference (univariate − full): -0.101
{'label': 'Univariate LR (initial_deviation_pct)', 'n_folds': 25, 'f1_mean': np.float64(0.314), 'f1_std': np.float64(0.184), 'recall_mean': np.float64(0.299), 'recall_std': np.float64(0.198), 'prec_mean': np.float64(0.35), 'auc_mean': np.float64(0.765), 'auc_std': np.float64(0.084), 'avg_n_feats': np.float64(1.0)}


In [16]:
# ── STAGE 4C: ODDS RATIO — Marginal effect of initial_deviation_pct ──
# Full-sample logistic regression (no train/test split).
# This is DESCRIPTIVE inference (coefficient estimation), not
# predictive evaluation — kept separate from the nested CV results above.
import statsmodels.api as sm

print("\n" + "=" * 65)
print("STAGE 4C: ODDS RATIO — initial_deviation_pct (full sample, n=135)")
print("=" * 65)

X_or = sm.add_constant(df[['initial_deviation_pct']])
y_or = df['y']

logit_model = sm.Logit(y_or, X_or).fit(disp=0)
print(logit_model.summary())

coef            = logit_model.params['initial_deviation_pct']
odds_ratio      = np.exp(coef)
ci_low, ci_high = np.exp(logit_model.conf_int().loc['initial_deviation_pct'])
pval            = logit_model.pvalues['initial_deviation_pct']

print(f"\nOdds Ratio per 1 percentage-point increase in initial deviation: "
      f"{odds_ratio:.3f}")
print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"p-value: {pval:.4f}")


STAGE 4C: ODDS RATIO — initial_deviation_pct (full sample, n=135)
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  135
Model:                          Logit   Df Residuals:                      133
Method:                           MLE   Df Model:                            1
Date:                Sat, 15 Aug 2026   Pseudo R-squ.:                 0.01208
Time:                        17:36:50   Log-Likelihood:                -68.085
converged:                       True   LL-Null:                       -68.918
Covariance Type:            nonrobust   LLR p-value:                    0.1970
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -1.5511      0.272     -5.707      0.000      -2.084      -1.018
initial_deviation_pct     0.052

In [17]:
# ── Paired bootstrap CI: LR vs RF F1 difference ──
# Uses the SAME 25 fold-level F1 scores already computed in Cell 12,
# no need to rerun the pipeline.
print("\n" + "=" * 65)
print("BOOTSTRAP CI: LR vs RF F1 difference (paired, same 25 folds)")
print("=" * 65)

lr_f1s = np.array([r['f1'] for r in primary_results['LR  class_weight']])
rf_f1s = np.array([r['f1'] for r in primary_results['RF  class_weight']])

diffs = lr_f1s - rf_f1s   # paired差異，因為兩者用的是同一組 25 折

n_boot = 10000
rng = np.random.default_rng(42)
boot_means = np.array([
    rng.choice(diffs, size=len(diffs), replace=True).mean()
    for _ in range(n_boot)
])

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"\nObserved F1 difference (LR - RF): {diffs.mean():.3f}")
print(f"95% Bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"CI includes zero: {ci_low <= 0 <= ci_high}")


BOOTSTRAP CI: LR vs RF F1 difference (paired, same 25 folds)

Observed F1 difference (LR - RF): 0.013
95% Bootstrap CI: [-0.056, 0.087]
CI includes zero: True


## STAGE 5: ROBUSTNESS CHECKS

In [ ]:
print("\n" + "=" * 65)
print("STAGE 5: ROBUSTNESS CHECKS")
print("=" * 65)

rob_summaries = []

# ── Robustness 1: Group-aware CV (contagion leakage) ─────────────────────────
print("\n--- Robustness 1: Group-aware CV ---")
gkf = GroupKFold(n_splits=5)
for name, mtype, _ in [('LR  class_weight', 'LR', False),
                        ('RF  class_weight', 'RF', False)]:
    res, _ = run_nested_cv(
        X_full, y, mtype=mtype, use_smote=False,
        outer_cv=gkf, groups_arr=groups)
    s = summarise(res, label=name + ' [group-aware]')
    rob_summaries.append(s)
    pf1 = [x['f1_mean'] for x in all_summaries if x['label']==name][0]
    delta = s['f1_mean'] - pf1
    verdict = "✓ stable" if abs(delta) < 0.05 else "~ modest"
    print(f"  {name:<20} F1={s['f1_mean']:.3f}  "
          f"ΔF1={delta:+.3f}  AUC={s['auc_mean']:.3f}  {verdict}")

# ── Robustness 2: 7-day cutoff ────────────────────────────────────────────────
print("\n--- Robustness 2: 7-day recovery cutoff ---")
y_7d = df['y_7d'].values
print(f"  slow (>7d): {y_7d.sum()}  fast (≤7d): {(y_7d==0).sum()}")
for name, mtype, _ in [('LR  class_weight', 'LR', False),
                        ('RF  class_weight', 'RF', False)]:
    res, _ = run_nested_cv(
        X_full, y_7d, mtype=mtype, use_smote=False,
        outer_cv=outer_cv)
    s = summarise(res, label=name + ' [7d cutoff]')
    rob_summaries.append(s)
    print(f"  {name:<20} F1={s['f1_mean']:.3f}±{s['f1_std']:.3f}  "
          f"AUC={s['auc_mean']:.3f}")

# ── Robustness 3: Exclude zero-slow coins ─────────────────────────────────────
print("\n--- Robustness 3: Exclude  FRAX / BUSD (zero slow events) ---")
df3 = df[~df['coin'].isin([ 'FRAX', 'BUSD'])].copy()
df3[CANDIDATE_POOL] = df3[CANDIDATE_POOL].fillna(
    df3[CANDIDATE_POOL].median())
X3, y3 = df3[CANDIDATE_POOL].values, df3['y'].values
print(f"  n={len(df3)}  slow={y3.sum()}  fast={(y3==0).sum()}")
outer3 = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
for name, mtype, _ in [('LR  class_weight', 'LR', False),
                        ('RF  class_weight', 'RF', False)]:
    res, _ = run_nested_cv(X3, y3, mtype=mtype, use_smote=False,
                           outer_cv=outer3)
    s = summarise(res, label=name + ' [excl. zero-slow]')
    rob_summaries.append(s)
    pf1 = [x['f1_mean'] for x in all_summaries if x['label']==name][0]
    delta = s['f1_mean'] - pf1
    print(f"  {name:<20} F1={s['f1_mean']:.3f}  "
          f"ΔF1={delta:+.3f}  AUC={s['auc_mean']:.3f}")


STAGE 5: ROBUSTNESS CHECKS

--- Robustness 1: Group-aware CV ---
  LR  class_weight     F1=0.246  ΔF1=-0.169  AUC=0.614  ~ modest
  RF  class_weight     F1=0.340  ΔF1=-0.062  AUC=0.705  ~ modest

--- Robustness 2: 7-day recovery cutoff ---
  slow (>7d): 11  fast (≤7d): 124
  LR  class_weight     F1=0.172±0.156  AUC=0.540
  RF  class_weight     F1=0.205±0.184  AUC=0.642

--- Robustness 3: Exclude  FRAX / BUSD (zero slow events) ---
  n=127  slow=28  fast=99
  LR  class_weight     F1=0.402  ΔF1=-0.013  AUC=0.672
  RF  class_weight     F1=0.417  ΔF1=+0.015  AUC=0.719


## STAGE 5B: SENSITIVITY ANALYSIS — Direct H1 test

In [18]:
# ══════════════════════════════════════════════════════════════════
# ROBUSTNESS 4 (NEW): DIRECT H1 TEST
# On-chain Collateral (DAI, LUSD) vs Algorithmic (FRAX, USDD, USTC)
# ------------------------------------------------------------------
# Rationale:
#   The primary nested-CV result shows type_algo/type_onchain were
#   NEVER selected across 25 folds. But this could mean either:
#     (a) mechanism type genuinely carries no independent signal, or
#     (b) the signal is crowded out because the 4-way mechanism
#         comparison is diluted by BUSD (n=1, contributes nothing)
#         and by tokenised funds acting as the reference category,
#         while initial_deviation_pct dominates feature selection.
#   This block isolates the *cleanest* theoretical contrast implied
#   by the arbitrage framework (Section 2.5): collateral-backed vs
#   no-collateral (algorithmic) redemption channels. Tokenised funds
#   and BUSD (off-chain, n=1) are dropped entirely so the comparison
#   is not diluted by categories irrelevant to this specific contrast.
#
# Two complementary tests are run:
#   4a. Unconditional association  — Fisher's exact test on the
#       2x2 table (mechanism type x fast/slow), i.e. raw H1 with no
#       controls.
#   4b. Conditional association    — logistic regression
#       y ~ type_algo + initial_deviation_pct, i.e. does mechanism
#       type still matter once the dominant confound (shock
#       severity) is controlled for. This is the test that most
#       directly answers "is H1 truly null, or was it masked?"
#   4c. Restricted nested CV (supplementary, small-sample caveat)
#       — re-run the existing nested CV pipeline on this subsample
#       only, to see whether type_algo's selection frequency rises
#       once tokenised-fund/BUSD noise is removed.
#
# Insert this cell AFTER Stage 5 (Robustness Checks) in the existing
# notebook. It reuses df, CANDIDATE_POOL, run_nested_cv, summarise,
# and RepeatedStratifiedKFold already defined earlier in the pipeline.
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from sklearn.model_selection import RepeatedStratifiedKFold

print("\n" + "=" * 65)
print("ROBUSTNESS 4: DIRECT H1 TEST — On-chain Collateral vs Algorithmic")
print("=" * 65)

# ── Build subsample: keep only on-chain collateral or algorithmic ──
df_h1 = df[(df['type_onchain'] == 1) | (df['type_algo'] == 1)].copy()
df_h1[CANDIDATE_POOL] = df_h1[CANDIDATE_POOL].fillna(
    df_h1[CANDIDATE_POOL].median())

n_total   = len(df_h1)
n_onchain = (df_h1['type_onchain'] == 1).sum()
n_algo    = (df_h1['type_algo'] == 1).sum()
slow_onchain = df_h1.loc[df_h1['type_onchain'] == 1, 'y'].sum()
slow_algo    = df_h1.loc[df_h1['type_algo'] == 1, 'y'].sum()

print(f"\nSubsample: n={n_total} "
      f"(on-chain={n_onchain}, algorithmic={n_algo})")
print(f"  On-chain collateral : slow={slow_onchain}/{n_onchain} "
      f"({slow_onchain/n_onchain*100:.1f}%)")
print(f"  Algorithmic         : slow={slow_algo}/{n_algo} "
      f"({slow_algo/n_algo*100:.1f}%)")

# ── 4a. Fisher's exact test (unconditional, raw H1) ─────────────────
table = [
    [slow_onchain, n_onchain - slow_onchain],   # on-chain: slow, fast
    [slow_algo,    n_algo - slow_algo],         # algo:     slow, fast
]
odds_ratio, p_fisher = stats.fisher_exact(table, alternative='two-sided')

print("\n--- 4a. Fisher's exact test (unconditional) ---")
print(f"  2x2 table (rows=mechanism, cols=[slow, fast]): {table}")
print(f"  Odds ratio = {odds_ratio:.3f}   p = {p_fisher:.3f}")
print(f"  Interpretation: {'no significant' if p_fisher > 0.05 else 'significant'} "
      f"unconditional association between mechanism type and recovery speed "
      f"at α=0.05.")

# ── 4b. Logistic regression controlling for initial_deviation_pct ──
X_h1 = df_h1[['type_algo', 'initial_deviation_pct']].copy()
X_h1 = sm.add_constant(X_h1)
y_h1 = df_h1['y'].values

logit_model = sm.Logit(y_h1, X_h1).fit(disp=0)

print("\n--- 4b. Logistic regression: y ~ type_algo + initial_deviation_pct ---")
print(logit_model.summary2().tables[1])

or_algo = np.exp(logit_model.params['type_algo'])
ci_low, ci_high = np.exp(logit_model.conf_int().loc['type_algo'])
p_algo = logit_model.pvalues['type_algo']

print(f"\n  type_algo: OR = {or_algo:.3f}  "
      f"95% CI = [{ci_low:.3f}, {ci_high:.3f}]  p = {p_algo:.3f}")
print(f"  Interpretation: after controlling for initial deviation severity, "
      f"mechanism type is {'NOT' if p_algo > 0.05 else ''} "
      f"significantly associated with recovery speed in this subsample.")

# ── 4c. Restricted nested CV (supplementary; n is small — caveat) ──
print("\n--- 4c. Restricted nested CV on subsample (supplementary check) ---")
print(f"  NOTE: n={n_total}, slow={int(y_h1.sum())}. With 5-fold CV each test "
      f"fold holds only ~3-4 slow events — treat this as indicative only, "
      f"not a stable estimate.")

X_h1_full = df_h1[CANDIDATE_POOL].values
outer_h1 = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

h1_results, h1_feats = run_nested_cv(
    X_h1_full, y_h1, mtype='LR', use_smote=False,
    outer_cv=outer_h1, label='LR [H1 subsample]')
s_h1 = summarise(h1_results, label='LR [on-chain vs algo subsample]')

print(f"  F1={s_h1['f1_mean']:.3f}±{s_h1['f1_std']:.3f}  "
      f"AUC={s_h1['auc_mean']:.3f}  avg_n_feats={s_h1['avg_n_feats']:.1f}")

total_folds_h1 = len(h1_feats)
freq_h1 = {}
for fold_list in h1_feats:
    for f in fold_list:
        freq_h1[f] = freq_h1.get(f, 0) + 1

type_algo_freq = freq_h1.get('type_algo', 0) / total_folds_h1 * 100
print(f"  type_algo selection frequency in this subsample: "
      f"{type_algo_freq:.0f}% of folds "
      f"(vs 0% in the full 9-coin sample, Section 4.2)")

# ── Summary table for write-up ──────────────────────────────────────
h1_summary_table = pd.DataFrame([{
    'test': "Fisher's exact (unconditional)",
    'statistic': f"OR={odds_ratio:.3f}",
    'p_value': round(p_fisher, 3),
    'n': n_total,
}, {
    'test': "Logistic regression (controlling for initial_deviation_pct)",
    'statistic': f"OR={or_algo:.3f} [{ci_low:.3f}, {ci_high:.3f}]",
    'p_value': round(p_algo, 3),
    'n': n_total,
}, {
    'test': "Nested CV type_algo selection frequency",
    'statistic': f"{type_algo_freq:.0f}% of folds",
    'p_value': np.nan,
    'n': n_total,
}])

print("\n--- Summary table (for Section 3.9 / 5.1) ---")
print(h1_summary_table.to_string(index=False))

h1_summary_table.to_csv('h1_direct_subsample_test.csv', index=False)
print("\nSaved: h1_direct_subsample_test.csv")


ROBUSTNESS 4: DIRECT H1 TEST — On-chain Collateral vs Algorithmic

Subsample: n=84 (on-chain=56, algorithmic=28)
  On-chain collateral : slow=11/56 (19.6%)
  Algorithmic         : slow=7/28 (25.0%)

--- 4a. Fisher's exact test (unconditional) ---
  2x2 table (rows=mechanism, cols=[slow, fast]): [[np.int64(11), np.int64(45)], [np.int64(7), np.int64(21)]]
  Odds ratio = 0.733   p = 0.583
  Interpretation: no significant unconditional association between mechanism type and recovery speed at α=0.05.

--- 4b. Logistic regression: y ~ type_algo + initial_deviation_pct ---
                          Coef.  Std.Err.         z     P>|z|    [0.025  \
const                 -3.828643  0.995850 -3.844597  0.000121 -5.780473   
type_algo              0.830629  0.613634  1.353622  0.175857 -0.372072   
initial_deviation_pct  0.715660  0.264143  2.709369  0.006741  0.197950   

                         0.975]  
const                 -1.876812  
type_algo              2.033329  
initial_deviation_pct  

In [21]:
liquidity_features = ['sc_ret_1d', 'sc_ret_7d', 'sc_ret_30d', 
                       'sc_vol_chg_1d', 'sc_vol_chg_7d', 'sc_vol_chg_30d',
                       'sc_mcap_chg_1d', 'sc_mcap_chg_7d', 'sc_vol_ratio']

corr_with_deviation = df[liquidity_features + ['initial_deviation_pct']].corr()['initial_deviation_pct'].drop('initial_deviation_pct')
print(corr_with_deviation.sort_values(key=abs, ascending=False))

sc_ret_1d        -0.342622
sc_vol_chg_7d     0.087540
sc_vol_chg_30d    0.068207
sc_mcap_chg_1d   -0.055988
sc_ret_7d         0.040119
sc_ret_30d        0.024101
sc_vol_chg_1d    -0.017018
sc_mcap_chg_7d    0.002194
sc_vol_ratio      0.000712
Name: initial_deviation_pct, dtype: float64


## STAGE 6: EDA & DESCRIPTIVE STATISTICS

In [ ]:
print("\n" + "=" * 65)
print("STAGE 6: EDA & DESCRIPTIVE STATISTICS")
print("=" * 65)

# Duration distribution
dur = df['duration_days']
print(f"\nDuration distribution:")
print(f"  Min={dur.min()}  P25={dur.quantile(.25):.0f}  "
      f"Median={dur.median():.0f}  P75={dur.quantile(.75):.0f}  "
      f"P90={dur.quantile(.9):.0f}  Max={dur.max()}")
print(f"  P75=3 confirms the recovery cutoff choice")

# Events by coin
print(f"\nEvents by coin:")
print(f"  {'Coin':<8} {'Type':<25} {'N':>4} {'Fast':>5} "
      f"{'Slow':>5} {'Slow%':>6}")
for coin in sorted(df['coin'].unique()):
    sub = df[df['coin']==coin]
    slow = sub['y'].sum()
    fast = (sub['y']==0).sum()
    mtype = sub['type'].iloc[0]
    print(f"  {coin:<8} {mtype:<25} {len(sub):>4} "
          f"{fast:>5} {slow:>5} {slow/len(sub)*100:>5.0f}%")

# Point-biserial correlations
print(f"\nPoint-biserial correlation with outcome (top 10):")
pb_rows = []
for feat in CANDIDATE_POOL:
    r, p = stats.pointbiserialr(y, df[feat])
    pb_rows.append({'feature': feat, 'r': round(r,3), 'p': round(p,4)})
pb_df = pd.DataFrame(pb_rows).sort_values('r', key=abs, ascending=False)
for _, row in pb_df.head(10).iterrows():
    sig = '***' if row['p']<0.001 else '**' if row['p']<0.01 \
          else '*' if row['p']<0.05 else 'ns'
    print(f"  {row['feature']:<30} r={row['r']:>7.3f}  "
          f"p={row['p']:.4f}  {sig}")
print("  (All ns — justifies ML approach over simple statistical tests)")


STAGE 6: EDA & DESCRIPTIVE STATISTICS

Duration distribution:
  Min=1  P25=1  Median=1  P75=3  P90=6  Max=115
  P75=3 confirms the recovery cutoff choice

Events by coin:
  Coin     Type                         N  Fast  Slow  Slow%
  BUSD     Off-chain Collateral         1     1     0     0%
  DAI      On-chain Collateral         23    18     5    22%
  FRAX     Algorithmic (Hybrid)         7     7     0     0%
  LUSD     On-chain Collateral         33    27     6    18%
  TUSD     Tokenised Fund              15    11     4    27%
  USDC     Tokenised Fund              11     9     2    18%
  USDD     Algorithmic                 14     9     5    36%
  USDT     Tokenised Fund              24    20     4    17%
  USTC     Algorithmic                  7     5     2    29%

Point-biserial correlation with outcome (top 10):
  sc_dw_dev_30d                  r=  0.154  p=0.0744  ns
  eth_ret_1d                     r= -0.129  p=0.1359  ns
  initial_deviation_pct          r=  0.124  p=0.1508 

In [ ]:
FEATURES = [
    'initial_deviation_pct', 'direction_below',
    'type_tokenised', 'type_offchain', 'type_onchain', 'type_algo',
    'sc_ret_1d', 'sc_ret_7d', 'sc_ret_30d',
    'sc_vol_chg_1d', 'sc_vol_chg_7d', 'sc_vol_chg_30d',
    'sc_mcap_chg_1d', 'sc_mcap_chg_7d', 'sc_vol_ratio',
    'sc_vol_5d', 'sc_vol_30d', 'sc_price_dev_5d', 'sc_price_dev_30d',
    'sc_dw_dev_5d', 'sc_dw_dev_30d',
    'btc_ret_1d', 'btc_ret_7d', 'btc_ret_30d',
    'btc_vol_chg_7d', 'btc_vol_chg_30d', 'btc_realized_vol',
    'eth_ret_1d', 'eth_ret_7d', 'eth_ret_30d',
    'eth_vol_chg_7d', 'eth_mcap_chg_7d', 'eth_realized_vol',
]
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())
 
fast_n = (df['label'] == 'fast').sum()
slow_n = (df['label'] == 'slow').sum()
total = fast_n + slow_n
COLORS = ['#2E86AB', '#E63946']
 
print(f"n={total}  fast={fast_n} ({fast_n/total*100:.1f}%)  "
      f"slow={slow_n} ({slow_n/total*100:.1f}%)  "
      f"ratio={fast_n/slow_n:.1f}:1")
# ════════════════════════════════════════════════════════
# 1. FAST vs SLOW RATIO (bar + pie)
# ════════════════════════════════════════════════════════

 
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([fast_n, slow_n], labels=['Fast', 'Slow'], autopct='%1.1f%%',
       colors=COLORS, startangle=90, explode=(0, 0.06),
       textprops={'fontsize': 12})
ax.set_title(f'Figure 1: Distribution of Recovery Outcome\n(Total N = {total})',
             fontsize=13)
plt.tight_layout()
plt.savefig('fig1_fast_vs_slow.png', dpi=200, bbox_inches='tight')
plt.close()

n=135  fast=107 (79.3%)  slow=28 (20.7%)  ratio=3.8:1


In [ ]:
# ════════════════════════════════════════════════════════
# 2. CLASS IMBALANCE SUMMARY
# ════════════════════════════════════════════════════════
ratio = fast_n / slow_n
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Fast\n(majority)', 'Slow\n(minority)'], [fast_n, slow_n],
              color=COLORS, width=0.5)
for bar, n in zip(bars, [fast_n, slow_n]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{n}',
            ha='center', fontsize=13, fontweight='bold')
ax.annotate(f'Imbalance ratio \u2248 {ratio:.1f} : 1',
            xy=(0.5, 0.92), xycoords='axes fraction', ha='center',
            fontsize=13, fontweight='bold', color='#333',
            bbox=dict(boxstyle='round,pad=0.4', fc='#FFF3CD', ec='#E0A800'))
ax.set_ylabel('Number of events')
ax.set_title('Class Imbalance in Depeg Recovery Outcomes')
ax.set_ylim(0, fast_n * 1.25)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig2_class_imbalance.png', dpi=200, bbox_inches='tight')
plt.close()

In [ ]:
# ════════════════════════════════════════════════════════
# 3. RECOVERY DURATION DISTRIBUTION (real histogram)
# ════════════════════════════════════════════════════════
dur = df['duration_days']
fig, ax = plt.subplots(figsize=(9, 5))
bins = [1, 2, 3, 4, 5, 6, 7, 10, 15, 20, 30, 50, 70, 100, 120]
ax.hist(dur, bins=bins, color='#2E86AB', edgecolor='white', alpha=0.85)
ax.axvline(dur.median(), color='#F4A261', linestyle='--', linewidth=2,
           label=f'Median = {dur.median():.0f}d')
ax.axvline(dur.quantile(.75), color='#E63946', linestyle='--', linewidth=2,
           label=f'P75 = {dur.quantile(.75):.0f}d (recovery cutoff)')
ax.set_xscale('log')
ax.set_xlabel('Recovery duration (days, log scale)')
ax.set_ylabel('Number of events')
ax.set_title(f'Figure 3: Recovery Duration Distribution '
             f'(n={len(dur)}, max={dur.max():.0f}d)')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig3_duration_distribution.png', dpi=200, bbox_inches='tight')
plt.close()

In [ ]:
#════════════════════════════════════════════════════════
# 4. FULL CORRELATION HEATMAP (33 features)
# ════════════════════════════════════════════════════════
corr = df[FEATURES].corr()
 
high_pairs = []
for i in range(len(FEATURES)):
    for j in range(i + 1, len(FEATURES)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_pairs.append((FEATURES[i], FEATURES[j], round(r, 3)))
high_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
 
print("\nHigh-correlation pairs (|r| > 0.7):")
for a, b, r in high_pairs:
    print(f"  {a:<25} <-> {b:<25} r={r:.3f}")
 
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES)))
ax.set_yticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=90, fontsize=7)
ax.set_yticklabels(FEATURES, fontsize=7)
for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        r = corr.iloc[i, j]
        if i != j and abs(r) > 0.7:
            ax.text(j, i, f'{r:.2f}', ha='center', va='center',
                    fontsize=6, color='white', fontweight='bold')
cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label('Pearson r')
ax.set_title('Figure 4: Correlation Matrix — 33 Candidate Features\n'
             '(white-bold labels = |r| > 0.7, flagged for multicollinearity removal)',
             fontsize=12)
plt.tight_layout()
plt.savefig('fig4_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.close()
 



High-correlation pairs (|r| > 0.7):
  eth_ret_7d                <-> eth_mcap_chg_7d           r=0.999
  sc_vol_30d                <-> sc_price_dev_30d          r=0.889
  btc_realized_vol          <-> eth_realized_vol          r=0.882
  btc_ret_7d                <-> eth_mcap_chg_7d           r=0.768
  btc_ret_7d                <-> eth_ret_7d                r=0.767
  btc_ret_30d               <-> eth_ret_30d               r=0.765
  btc_ret_1d                <-> eth_ret_1d                r=0.728
  sc_vol_5d                 <-> sc_vol_30d                r=0.716


In [ ]:
# ════════════════════════════════════════════════════════
# 5. FOCUSED HEATMAP — only features in a high-correlation pair
# ════════════════════════════════════════════════════════
involved = sorted(set([a for a, b, r in high_pairs] +
                       [b for a, b, r in high_pairs]))
corr_sub = corr.loc[involved, involved]
 
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_sub.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(involved)))
ax.set_yticks(range(len(involved)))
ax.set_xticklabels(involved, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(involved, fontsize=9)
for i in range(len(involved)):
    for j in range(len(involved)):
        r = corr_sub.iloc[i, j]
        color = 'white' if abs(r) > 0.5 else 'black'
        ax.text(j, i, f'{r:.2f}', ha='center', va='center',
                fontsize=8, color=color)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cbar.set_label('Pearson r')
ax.set_title('Figure 5: Features Involved in High-Correlation Pairs (|r| > 0.7)',
             fontsize=11)
plt.tight_layout()
plt.savefig('fig5_correlation_highpairs.png', dpi=200, bbox_inches='tight')
plt.close()
 

 

In [ ]:
# ════════════════════════════════════════════════════════
# SAVE CORRELATION TABLES (useful for dissertation appendix)
# ════════════════════════════════════════════════════════
pd.DataFrame(high_pairs, columns=['feature_a', 'feature_b', 'r']).to_csv(
    'high_correlation_pairs.csv', index=False)
corr.to_csv('full_correlation_matrix.csv')
 
print("\nSaved: fig1_fast_vs_slow.png, fig2_class_imbalance.png, "
      "fig3_duration_distribution.png, fig4_correlation_heatmap.png, "
      "fig5_correlation_highpairs.png, high_correlation_pairs.csv, "
      "full_correlation_matrix.csv")


Saved: fig1_fast_vs_slow.png, fig2_class_imbalance.png, fig3_duration_distribution.png, fig4_correlation_heatmap.png, fig5_correlation_highpairs.png, high_correlation_pairs.csv, full_correlation_matrix.csv


## STAGE 7: SAVE ALL OUTPUTS

In [ ]:
print("\n" + "=" * 65)
print("STAGE 7: SAVE ALL OUTPUTS")
print("=" * 65)

# Primary results
pd.DataFrame(all_summaries).to_csv('model_results_primary.csv', index=False)

# Robustness results
pd.DataFrame(rob_summaries).to_csv('model_results_robustness.csv', index=False)

# Feature selection frequency
freq_df.to_csv('feature_selection_frequency.csv', index=False)

# EDA outputs
pb_df.to_csv('eda_pointbiserial.csv', index=False)
df[CANDIDATE_POOL + ['coin','type','duration_days','y']]\
    .describe().to_csv('eda_descriptive_stats.csv')

# Multicollinearity summary
pd.DataFrame(high_pairs, columns=['feature_a','feature_b','r'])\
    .to_csv('multicollinearity_pairs.csv', index=False)
pd.DataFrame([{'feature':k, 'reason':v} for k,v in REMOVED.items()])\
    .to_csv('removed_features.csv', index=False)

print("\nSaved:")
print("  model_results_primary.csv       — Table 5: primary results")
print("  model_results_robustness.csv    — Table 6: robustness checks")
print("  feature_selection_frequency.csv — Table 7: feature stability")
print("  eda_pointbiserial.csv           — Table 3: EDA correlations")
print("  eda_descriptive_stats.csv       — Table 2: descriptive stats")
print("  multicollinearity_pairs.csv     — Appendix: high-corr pairs")
print("  removed_features.csv            — Appendix: removal decisions")


STAGE 7: SAVE ALL OUTPUTS

Saved:
  model_results_primary.csv       — Table 5: primary results
  model_results_robustness.csv    — Table 6: robustness checks
  feature_selection_frequency.csv — Table 7: feature stability
  eda_pointbiserial.csv           — Table 3: EDA correlations
  eda_descriptive_stats.csv       — Table 2: descriptive stats
  multicollinearity_pairs.csv     — Appendix: high-corr pairs
  removed_features.csv            — Appendix: removal decisions


In [ ]:
# =================================================================
# STAGE 6: OVERFITTING DIAGNOSTIC — Train vs Test F1/AUC Gap
# =================================================================

from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import cross_val_score, StratifiedKFold, RepeatedStratifiedKFold

def run_fold_with_train_score(X_tr, y_tr, X_te, y_te, mtype, seed):
    """
    與 run_single_fold 相同的 nested CV 邏輯,但額外回傳 train-fold 的
    F1/AUC,用來跟 test-fold 分數比較,診斷 overfitting 程度。
    """
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    sc = MinMaxScaler().fit(X_tr)
    X_tr_s = sc.transform(X_tr)

    rf_rank = RandomForestClassifier(
        n_estimators=50, max_depth=3, min_samples_leaf=5,
        class_weight='balanced', random_state=seed
    ).fit(X_tr_s, y_tr)
    fi = sorted(zip(CANDIDATE_POOL, rf_rank.feature_importances_),
                key=lambda x: x[1], reverse=True)

    best_n, best_inner_f1 = 1, -1
    best_C, best_depth, best_leaf = 10, 1, 10

    for n in range(1, min(MAX_N + 1, len(CANDIDATE_POOL) + 1)):
        fidx = [CANDIDATE_POOL.index(f) for f, _ in fi[:n]]
        X_n = X_tr_s[:, fidx]
        if mtype == 'LR':
            for C in LR_C_GRID:
                m = LogisticRegression(C=C, penalty='l2', solver='liblinear',
                                        class_weight='balanced', max_iter=300,
                                        random_state=seed)
                s = cross_val_score(m, X_n, y_tr, cv=inner_cv, scoring='f1').mean()
                if s > best_inner_f1:
                    best_inner_f1, best_n, best_C = s, n, C
        else:
            for depth in RF_GRID['max_depth']:
                for leaf in RF_GRID['min_samples_leaf']:
                    m = RandomForestClassifier(n_estimators=50, max_depth=depth,
                                                min_samples_leaf=leaf, max_features='sqrt',
                                                class_weight='balanced', random_state=seed)
                    s = cross_val_score(m, X_n, y_tr, cv=inner_cv, scoring='f1').mean()
                    if s > best_inner_f1:
                        best_inner_f1, best_n, best_depth, best_leaf = s, n, depth, leaf

    fidx = [CANDIDATE_POOL.index(f) for f, _ in fi[:best_n]]
    X_tr_f, X_te_f = X_tr_s[:, fidx], sc.transform(X_te)[:, fidx]

    if mtype == 'LR':
        model = LogisticRegression(C=best_C, penalty='l2', solver='liblinear',
                                    class_weight='balanced', max_iter=300,
                                    random_state=seed).fit(X_tr_f, y_tr)
    else:
        model = RandomForestClassifier(n_estimators=100, max_depth=best_depth,
                                        min_samples_leaf=best_leaf, max_features='sqrt',
                                        class_weight='balanced', random_state=seed).fit(X_tr_f, y_tr)

    train_f1 = f1_score(y_tr, model.predict(X_tr_f), zero_division=0)
    test_f1  = f1_score(y_te, model.predict(X_te_f), zero_division=0)
    train_auc = roc_auc_score(y_tr, model.predict_proba(X_tr_f)[:, 1])
    test_auc  = roc_auc_score(y_te, model.predict_proba(X_te_f)[:, 1])
    return train_f1, test_f1, train_auc, test_auc


print("=" * 65)
print("STAGE 6: Train vs Test Gap Diagnostic (25 outer folds)")
print("=" * 65)

X_full = df[CANDIDATE_POOL].values
y_full = df['y'].values if 'y' in df.columns else (df['label'].astype(str).str.lower() == 'slow').astype(int).values

for mtype in ['LR', 'RF']:
    outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    tr_f1s, te_f1s, tr_aucs, te_aucs = [], [], [], []
    for i, (tr, te) in enumerate(outer.split(X_full, y_full)):
        seed = i % 5
        tf1, ef1, tauc, eauc = run_fold_with_train_score(
            X_full[tr], y_full[tr], X_full[te], y_full[te], mtype, seed)
        tr_f1s.append(tf1); te_f1s.append(ef1)
        tr_aucs.append(tauc); te_aucs.append(eauc)

    print(f"\n{mtype} class_weight:")
    print(f"  Train F1: {np.mean(tr_f1s):.3f}   Test F1: {np.mean(te_f1s):.3f} ± {np.std(te_f1s):.3f}   "
          f"Gap: {np.mean(tr_f1s) - np.mean(te_f1s):.3f}")
    print(f"  Train AUC: {np.mean(tr_aucs):.3f}   Test AUC: {np.mean(te_aucs):.3f}   "
          f"Gap: {np.mean(tr_aucs) - np.mean(te_aucs):.3f}")

STAGE 6: Train vs Test Gap Diagnostic (25 outer folds)

LR class_weight:
  Train F1: 0.449   Test F1: 0.353 ± 0.196   Gap: 0.096
  Train AUC: 0.739   Test AUC: 0.659   Gap: 0.080

RF class_weight:
  Train F1: 0.580   Test F1: 0.391 ± 0.147   Gap: 0.189
  Train AUC: 0.848   Test AUC: 0.715   Gap: 0.134


In [ ]:
# =================================================================
# STAGE 6b: Regularisation Strength Comparison (RF only)
# =================================================================
configs = {
    'weak   (depth 2-3, leaf 5-7, all features)': ([2, 3], [5, 7], [None], 100),
    'medium (depth 1-2, leaf 10-15, sqrt)':       ([1, 2], [10, 15], ['sqrt'], 100),
    'strong (depth 1, leaf 15-20, sqrt, n=50)':   ([1], [15, 20], ['sqrt'], 50),
}

def run_fold_configurable(X_tr, y_tr, X_te, y_te, seed, depth_grid, leaf_grid, feat_grid, n_est):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    sc = MinMaxScaler().fit(X_tr)
    X_tr_s = sc.transform(X_tr)
    rf_rank = RandomForestClassifier(n_estimators=50, max_depth=3, min_samples_leaf=5,
                                      class_weight='balanced', random_state=seed).fit(X_tr_s, y_tr)
    fi = sorted(zip(CANDIDATE_POOL, rf_rank.feature_importances_), key=lambda x: x[1], reverse=True)

    best_n, best_inner = 1, -1
    best_depth, best_leaf, best_feat = depth_grid[0], leaf_grid[0], feat_grid[0]
    for n in range(1, min(MAX_N + 1, len(CANDIDATE_POOL) + 1)):
        fidx = [CANDIDATE_POOL.index(f) for f, _ in fi[:n]]
        X_n = X_tr_s[:, fidx]
        for depth in depth_grid:
            for leaf in leaf_grid:
                for feat in feat_grid:
                    m = RandomForestClassifier(n_estimators=50, max_depth=depth,
                                                min_samples_leaf=leaf, max_features=feat,
                                                class_weight='balanced', random_state=seed)
                    s = cross_val_score(m, X_n, y_tr, cv=inner_cv, scoring='f1').mean()
                    if s > best_inner:
                        best_inner, best_n, best_depth, best_leaf, best_feat = s, n, depth, leaf, feat

    fidx = [CANDIDATE_POOL.index(f) for f, _ in fi[:best_n]]
    X_tr_f, X_te_f = X_tr_s[:, fidx], sc.transform(X_te)[:, fidx]
    model = RandomForestClassifier(n_estimators=n_est, max_depth=best_depth,
                                    min_samples_leaf=best_leaf, max_features=best_feat,
                                    class_weight='balanced', random_state=seed).fit(X_tr_f, y_tr)
    train_f1 = f1_score(y_tr, model.predict(X_tr_f), zero_division=0)
    test_f1  = f1_score(y_te, model.predict(X_te_f), zero_division=0)
    return train_f1, test_f1

print("=" * 65)
print("STAGE 6b: Regularisation Strength vs Test F1 (25 outer folds)")
print("=" * 65)

outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
splits = list(outer.split(X_full, y_full))

for name, (depth_grid, leaf_grid, feat_grid, n_est) in configs.items():
    tr_f1s, te_f1s = [], []
    for i, (tr, te) in enumerate(splits):
        seed = i % 5
        tf1, ef1 = run_fold_configurable(X_full[tr], y_full[tr], X_full[te], y_full[te],
                                          seed, depth_grid, leaf_grid, feat_grid, n_est)
        tr_f1s.append(tf1); te_f1s.append(ef1)
    print(f"\n{name}:")
    print(f"  Train F1: {np.mean(tr_f1s):.3f}   Test F1: {np.mean(te_f1s):.3f} ± {np.std(te_f1s):.3f}   "
          f"Gap: {np.mean(tr_f1s) - np.mean(te_f1s):.3f}")

STAGE 6b: Regularisation Strength vs Test F1 (25 outer folds)

weak   (depth 2-3, leaf 5-7, all features):
  Train F1: 0.657   Test F1: 0.409 ± 0.167   Gap: 0.248

medium (depth 1-2, leaf 10-15, sqrt):
  Train F1: 0.580   Test F1: 0.391 ± 0.147   Gap: 0.189

strong (depth 1, leaf 15-20, sqrt, n=50):
  Train F1: 0.559   Test F1: 0.382 ± 0.112   Gap: 0.177
